## Se cargan los videos obtenidos

### Configuración

In [1]:
import os, shutil
print("ffmpeg  ->", shutil.which("ffmpeg"))
print("ffprobe ->", shutil.which("ffprobe"))

# Fallback/forzado con imageio-ffmpeg (opcional)
try:
    import imageio_ffmpeg
    os.environ["IMAGEIO_FFMPEG_EXE"] = imageio_ffmpeg.get_ffmpeg_exe()
    print("IMAGEIO_FFMPEG_EXE ->", os.environ["IMAGEIO_FFMPEG_EXE"])
except Exception as e:
    print("imageio-ffmpeg no disponible:", e)

# Configurar pydub (opcional)
try:
    from pydub import AudioSegment
    if shutil.which("ffmpeg"):  AudioSegment.converter = shutil.which("ffmpeg")
    if shutil.which("ffprobe"): AudioSegment.ffprobe   = shutil.which("ffprobe")
    print("pydub OK")
except Exception as e:
    print("pydub error:", e)


ffmpeg  -> C:\Users\RICHI\anaconda3\envs\TesisRichiFix\Library\bin\ffmpeg.EXE
ffprobe -> C:\Users\RICHI\anaconda3\envs\TesisRichiFix\Library\bin\ffprobe.EXE
IMAGEIO_FFMPEG_EXE -> c:\Users\RICHI\anaconda3\envs\TesisRichiFix\Library\bin\ffmpeg.exe
pydub OK


In [2]:
# Paths y parámetros
from pathlib import Path

INPUT_DIR  = Path(r"C:\Users\RICHI\Documents\Tesis\Data\Andinas\Original")   # cambia esto
OUTPUT_DIR = Path(r"C:\Users\RICHI\Documents\Tesis\Data\Andinas\Segmentacion_1min")    # cambia esto
TARGET_SECONDS = 60                     # longitud del corte
OUTPUT_SUFFIX = "_mid1min"              # sufijo del archivo de salida

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [3]:
!ffmpeg -version
!ffprobe -version

ffmpeg version 8.0 Copyright (c) 2000-2025 the FFmpeg developers
built with clang version 21.1.2
configuration: --prefix=/d/bld/ffmpeg_1758923948582/_h_env/Library --cc=clang.exe --cxx=clang++.exe --nm=llvm-nm --ar=llvm-ar --disable-doc --enable-openssl --enable-demuxer=dash --enable-hardcoded-tables --enable-libfreetype --enable-libharfbuzz --enable-libfontconfig --enable-libopenh264 --enable-libdav1d --ld=lld-link --target-os=win64 --enable-cross-compile --toolchain=msvc --host-cc=clang.exe --extra-libs=ucrt.lib --extra-libs=vcruntime.lib --extra-libs=oldnames.lib --strip=llvm-strip --disable-stripping --host-extralibs= --disable-libopenvino --enable-gpl --enable-libx264 --enable-libx265 --enable-libmp3lame --enable-libaom --enable-libsvtav1 --enable-libxml2 --enable-pic --enable-shared --disable-static --enable-version3 --enable-zlib --enable-libvorbis --enable-libopus --enable-librsvg --enable-ffplay --enable-vulkan --enable-libshaderc --pkg-config=/d/bld/ffmpeg_1758923948582/_buil

### Procesamiento por lotes

In [4]:
from pathlib import Path
from collections import defaultdict

EXTS = (".m4a", ".opus", ".aac", ".ogg", ".mp4")

def list_audio_recursive(root: Path):
    return sorted([p for p in root.rglob("*") if p.suffix.lower() in EXTS and p.is_file()])

# Sanitiza el nombre de género para integrarlo al filename: quita espacios y caracteres inválidos en Windows.
def genre_token(name: str) -> str:
    invalid = set('<>:"/\\|?*')
    tok = "".join(ch for ch in name if ch not in invalid)
    tok = "".join(tok.split())  # elimina espacios
    return tok

# Género = primer directorio debajo de INPUT_DIR
def genre_of(path: Path, input_root: Path) -> str:
    rel = path.parent.relative_to(input_root)
    return rel.parts[0] if len(rel.parts) >= 1 else "Genero"

def out_dir_for(in_path: Path, input_root: Path, output_root: Path) -> Path:
    rel_parent = in_path.parent.relative_to(input_root)   # replica subcarpetas
    out_dir = output_root / rel_parent
    out_dir.mkdir(parents=True, exist_ok=True)
    return out_dir

def next_available_name(out_dir: Path, base_token: str, start_idx: int, ext: str, suffix: str) -> tuple[str, int]:
    i = max(1, start_idx)
    while True:
        candidate = f"{base_token}{i}{suffix}{ext}"
        if not (out_dir / candidate).exists():
            return candidate, i
        i += 1

import subprocess, shutil
from pathlib import Path

# --- códec / formato de salida ---
def codec_for_ext(in_ext: str) -> tuple[str, list[str]]:
    """
    Unifica TODO a WAV PCM 16 kHz mono (recomendado para ML).
    Retorna (extensión_salida, args_ffmpeg_de_audio).
    """
    ext_out = ".wav"
    codec_args = ["-ac", "1", "-ar", "16000", "-sample_fmt", "s16", "-c:a", "pcm_s16le"]
    return ext_out, codec_args

# Si prefieres NO recodificar (rápido pero heterogéneo):
# def codec_for_ext(in_ext: str) -> tuple[str, list[str]]:
#     ext_out = in_ext.lower() if in_ext else ".m4a"
#     return ext_out, ["-c:a", "copy"]

# --- duración con ffprobe (robusto) ---
def get_duration_seconds(in_path: Path) -> float:
    """
    Intenta primero 'format=duration' y luego 'stream=duration'.
    Lanza ValueError si no se puede determinar.
    """
    in_path = str(in_path)

    if shutil.which("ffprobe") is None:
        raise RuntimeError("ffprobe no está en PATH.")

    # 1) duración del contenedor
    cmd1 = [
        "ffprobe", "-v", "error", "-show_entries", "format=duration",
        "-of", "default=nw=1:nk=1", in_path
    ]
    p1 = subprocess.run(cmd1, stdout=subprocess.PIPE, stderr=subprocess.PIPE)
    out = p1.stdout.decode("utf-8", "ignore").strip()
    try:
        dur = float(out)
        if dur > 0:
            return dur
    except Exception:
        pass

    # 2) duración del primer stream de audio
    cmd2 = [
        "ffprobe", "-v", "error", "-select_streams", "a:0",
        "-show_entries", "stream=duration",
        "-of", "default=nw=1:nk=1", in_path
    ]
    p2 = subprocess.run(cmd2, stdout=subprocess.PIPE, stderr=subprocess.PIPE)
    out2 = p2.stdout.decode("utf-8", "ignore").strip()
    try:
        dur2 = float(out2)
        if dur2 > 0:
            return dur2
    except Exception:
        pass

    raise ValueError("Duración no disponible para: " + in_path)

# --- construcción de comando ffmpeg para “mid-cut” ---
def midcut_command_args(in_path: Path, out_path: Path, dur_total: float, target_seconds: int, codec_args: list[str]) -> list[str]:
    """
    Extrae un segmento centrado. Si el archivo es más corto que target, toma todo.
    Usa seek antes de -i si el inicio > 0.5s para acelerar.
    """
    in_path = str(in_path)
    out_path = str(out_path)

    if shutil.which("ffmpeg") is None:
        raise RuntimeError("ffmpeg no está en PATH.")

    t = max(0.1, min(float(target_seconds), float(dur_total)))
    start = max(0.0, (float(dur_total) - t) / 2.0)

    # Para aceleración, seek de entrada sólo si es significativo
    args = ["ffmpeg", "-y"]
    if start >= 0.5:
        args += ["-ss", f"{start:.3f}", "-i", in_path, "-t", f"{t:.3f}"]
    else:
        args += ["-i", in_path, "-ss", f"{start:.3f}", "-t", f"{t:.3f}"]

    # sin vídeo
    args += ["-vn"]
    # parámetros de audio según política
    args += codec_args
    # salida temporal
    args += [out_path]
    return args

# --- wrapper de ejecución (captura stderr/stdout) ---
class Completed:
    __slots__ = ("returncode", "stdout", "stderr")
    def __init__(self, rc, so, se):
        self.returncode = rc
        self.stdout = so
        self.stderr = se

def run(args: list[str]) -> Completed:
    p = subprocess.run(args, stdout=subprocess.PIPE, stderr=subprocess.PIPE)
    return Completed(p.returncode, p.stdout, p.stderr)


def process_all_numbered_by_genre(input_root: Path, output_root: Path, target_seconds: int, suffix: str):
    files = list_audio_recursive(input_root)
    print(f"Encontrados: {len(files)} archivos.")
    counters = defaultdict(int)  # por género

    ok = fail = 0
    for f in files:
        try:
            gen = genre_of(f, input_root)
            gen_tok = genre_token(gen) or "Genero"
            out_dir = out_dir_for(f, input_root, output_root)
            ext_out, codec_args = codec_for_ext(f.suffix)

            # índice siguiente para este género, garantizando no colisión si ya existen archivos
            next_idx = counters[gen] + 1
            filename, assigned_idx = next_available_name(out_dir, gen_tok, next_idx, ext_out, suffix)

            dur = get_duration_seconds(f)
            out_tmp = (out_dir / filename).with_suffix(".tmp" + ext_out)
            args = midcut_command_args(f, out_tmp, dur, target_seconds, codec_args)
            p = run(args)
            if p.returncode != 0:
                fail += 1
                print(f"[ERROR] {f.name}: {p.stderr.strip()[:300]}")
                continue

            final_path = out_dir / filename
            if final_path.exists():
                final_path.unlink()
            out_tmp.rename(final_path)

            counters[gen] = max(counters[gen], assigned_idx)
            ok += 1
        except Exception as e:
            fail += 1
            print(f"[ERROR] {f}: {str(e)[:300]}")

    print(f"Éxitos: {ok} | Errores: {fail} | Carpeta salida: {output_root}")

# Ejecutar
process_all_numbered_by_genre(INPUT_DIR, OUTPUT_DIR, TARGET_SECONDS, OUTPUT_SUFFIX)



Encontrados: 1000 archivos.
Éxitos: 1000 | Errores: 0 | Carpeta salida: C:\Users\RICHI\Documents\Tesis\Data\Andinas\Segmentacion_1min


### Segmentación a 20 seg

In [5]:
from pathlib import Path
from typing import List
import shutil

# --- BINARIO ---
FFMPEG = shutil.which("ffmpeg") or "ffmpeg"

EXTS = (".m4a", ".opus", ".aac", ".ogg", ".mp4", ".wav")
SEGMENT_SECONDS = 20
SEGMENT_STARTS  = [0, 20, 40]
NAME_TAG_PREFIX = "t"
NAME_TAG_WITH_S = True

def list_audio_recursive(root: Path) -> List[Path]:
    root = Path(root)
    return sorted([p for p in root.rglob("*") if p.is_file() and p.suffix.lower() in EXTS])

def ensure_out_dir_for(in_path: Path, input_root: Path, output_root: Path) -> Path:
    rel = in_path.parent.relative_to(input_root)
    out_dir = output_root / rel
    out_dir.mkdir(parents=True, exist_ok=True)
    return out_dir

def tag_for_start(start: int) -> str:
    tag = f"{NAME_TAG_PREFIX}{start:03d}"
    return f"_{tag}" + ("s" if NAME_TAG_WITH_S else "")

def segment_command_args(inp: Path, out: Path, start_sec: float, seg_len: int, codec_args: list[str]) -> list[str]:
    return [
        FFMPEG, "-hide_banner", "-y",
        "-ss", f"{start_sec:.6f}",
        "-i", str(inp),
        "-t", f"{seg_len:.3f}",
        "-vn", "-map", "a:0",
        "-af", f"apad=whole_dur={seg_len}",
        *codec_args,
        str(out),
    ]

def process_minute_to_20s(input_root: Path, output_root: Path, *, verbose: bool = True, print_every: int = 10):
    # comprobaciones de dependencias para evitar “silencios”
    missing = []
    for fn in ("codec_for_ext", "get_duration_seconds", "run"):
        if fn not in globals():
            missing.append(fn)
    if missing:
        raise RuntimeError(f"Faltan helpers: {', '.join(missing)}. Define/importa antes de llamar.")

    pr = (lambda *a, **k: print(*a, **k, flush=True)) if verbose else (lambda *a, **k: None)

    input_root = Path(input_root)
    output_root = Path(output_root)
    output_root.mkdir(parents=True, exist_ok=True)

    pr("[INFO] Inicio segmentación 60s→20s")
    pr(f"[INFO] INPUT: {input_root}")
    pr(f"[INFO] OUTPUT: {output_root}")
    if not shutil.which(FFMPEG):
        raise OSError("ffmpeg no está en PATH")

    files = list_audio_recursive(input_root)
    pr(f"Entradas: {len(files)}")
    if not files:
        pr(f"[WARN] No se encontraron archivos con extensiones: {', '.join(EXTS)}")
        return

    ok = fail = 0
    for idx, f in enumerate(files, 1):
        try:
            dur = get_duration_seconds(f)
            out_dir = ensure_out_dir_for(f, input_root, output_root)
            ext_out, codec_args = codec_for_ext(f.suffix)

            for start in SEGMENT_STARTS:
                if start >= dur:
                    pr(f"[SKIP] {f.name} inicio {start}s > duración {dur:.3f}s")
                    continue

                tag = tag_for_start(int(start))
                out_final = out_dir / f"{f.stem}{tag}{ext_out}"
                out_tmp   = out_dir / f"{f.stem}{tag}.tmp{ext_out}"

                args = segment_command_args(f, out_tmp, float(start), SEGMENT_SECONDS, codec_args)
                pr(f"[RUN] {f.name} → {out_final.name}  ({start:>2}s…{start+SEGMENT_SECONDS}s)")
                p = run(args)

                rc = getattr(p, "returncode", 1)
                if rc != 0:
                    fail += 1
                    err = getattr(p, "stderr", b"")
                    err_msg = err.decode("utf-8", "ignore") if isinstance(err, (bytes, bytearray)) else str(err)
                    pr(f"[ERROR] {f.name} ({start}s): {err_msg.strip()[:400]}")
                    if out_tmp.exists():
                        out_tmp.unlink(missing_ok=True)
                    continue

                if out_final.exists():
                    out_final.unlink()
                out_tmp.rename(out_final)
                ok += 1

        except Exception as e:
            fail += 1
            pr(f"[ERROR] {f}: {str(e)[:400]}")

        # progreso periódico para notebooks que recortan la salida
        if verbose and (idx % print_every == 0):
            pr(f"[PROGRESS] Procesados {idx}/{len(files)} | OK {ok} | ERR {fail}")

    pr(f"Segmentos creados: {ok} | Errores: {fail} | Salida: {output_root}")
    pr("[INFO] Fin segmentación")



In [6]:
INPUT_DIR_MINUTE = Path(r"C:\Users\RICHI\Documents\Tesis\Data\Andinas\Segmentacion_1min")
OUTPUT_DIR_20S   = Path(r"C:\Users\RICHI\Documents\Tesis\Data\Andinas\Segmentacion20seg")

process_minute_to_20s(INPUT_DIR_MINUTE, OUTPUT_DIR_20S, verbose=True, print_every=10)

[INFO] Inicio segmentación 60s→20s
[INFO] INPUT: C:\Users\RICHI\Documents\Tesis\Data\Andinas\Segmentacion_1min
[INFO] OUTPUT: C:\Users\RICHI\Documents\Tesis\Data\Andinas\Segmentacion20seg
Entradas: 1000
[RUN] Chimayche100_mid1min.wav → Chimayche100_mid1min_t000s.wav  ( 0s…20s)
[RUN] Chimayche100_mid1min.wav → Chimayche100_mid1min_t020s.wav  (20s…40s)
[RUN] Chimayche100_mid1min.wav → Chimayche100_mid1min_t040s.wav  (40s…60s)
[RUN] Chimayche10_mid1min.wav → Chimayche10_mid1min_t000s.wav  ( 0s…20s)
[RUN] Chimayche10_mid1min.wav → Chimayche10_mid1min_t020s.wav  (20s…40s)
[RUN] Chimayche10_mid1min.wav → Chimayche10_mid1min_t040s.wav  (40s…60s)
[RUN] Chimayche11_mid1min.wav → Chimayche11_mid1min_t000s.wav  ( 0s…20s)
[RUN] Chimayche11_mid1min.wav → Chimayche11_mid1min_t020s.wav  (20s…40s)
[RUN] Chimayche11_mid1min.wav → Chimayche11_mid1min_t040s.wav  (40s…60s)
[RUN] Chimayche12_mid1min.wav → Chimayche12_mid1min_t000s.wav  ( 0s…20s)
[RUN] Chimayche12_mid1min.wav → Chimayche12_mid1min_t020s.wav

## Segmentacion 5seg

In [2]:
from pathlib import Path
import numpy as np
import librosa
from scipy.io.wavfile import write as wavwrite
from tqdm.auto import tqdm

# === Configura entradas/salidas ===
ORIG_ROOT = Path(r"C:\Users\RICHI\Documents\Tesis\Data\Andinas\Segmentacion_1min")  # <género>/<archivo_ext> (pistas completas)
SEG_ROOT  = Path(r"C:\Users\RICHI\Documents\Tesis\Data\Andina5seg\segmentacion5seg")      # salida: <género>/<TrackID>/<TrackID>_t000s.wav ...

SEG_DUR_SEC  = 5.0
OVERLAP_SEC  = 0.0  # pon 2.5 si quieres 50% solape
TARGET_SR    = None # None = deja el sr original; o pon 22050/44100 si quieres remuestrear

AUDIO_EXTS = {".wav", ".mp3", ".flac", ".ogg", ".m4a", ".aac"}

def float_to_int16(x: np.ndarray):
    x = np.clip(x, -1.0, 1.0)
    return (x * 32767.0).astype(np.int16)

def segment_track(in_path: Path, out_dir: Path, seg_len=5.0, hop=5.0, target_sr=None):
    """Segmenta una pista en trozos de seg_len (s), hop (s). Guarda WAV int16."""
    x, sr = librosa.load(str(in_path), sr=target_sr, mono=True)
    nseg = int(np.floor((len(x)/sr - seg_len) / hop)) + 1
    if nseg <= 0:
        return 0
    out_dir.mkdir(parents=True, exist_ok=True)
    stem = in_path.stem
    for i in range(nseg):
        t0 = int((i * hop) * sr)
        t1 = t0 + int(seg_len * sr)
        clip = x[t0:t1]
        # Nombre con sufijo temporal en segundos, rellenado a 3 dígitos (000,005,010...)
        tsec = int(i * hop)
        name = f"{stem}_t{tsec:03d}s.wav"
        y16 = float_to_int16(clip.astype(np.float32))
        wavwrite(str(out_dir / name), sr, y16)
    return nseg

def run_segmentation(orig_root: Path, seg_root: Path):
    total, created = 0, 0
    for gdir in sorted([d for d in orig_root.iterdir() if d.is_dir()]):
        genre = gdir.name
        files = [p for p in gdir.rglob("*") if p.suffix.lower() in AUDIO_EXTS]
        for track_path in tqdm(files, desc=f"Seg {genre}"):
            # Carpeta por TrackID = stem del archivo original
            track_id = track_path.stem
            out_dir = seg_root / genre / track_id
            n = segment_track(track_path, out_dir, seg_len=SEG_DUR_SEC, hop=max(1e-9, SEG_DUR_SEC-OVERLAP_SEC), target_sr=TARGET_SR)
            total += 1; created += (n > 0)
    print(f"Tracks procesados: {total} | con segmentos: {created}")

run_segmentation(ORIG_ROOT, SEG_ROOT)
print("Segmentos 5s en:", SEG_ROOT)


Seg Chimayche:   0%|          | 0/100 [00:00<?, ?it/s]

Seg Cumbia andina:   0%|          | 0/100 [00:00<?, ?it/s]

Seg Diablada:   0%|          | 0/100 [00:00<?, ?it/s]

Seg Huaylash:   0%|          | 0/100 [00:00<?, ?it/s]

Seg Huayno:   0%|          | 0/100 [00:00<?, ?it/s]

...

Seg Santiago:   0%|          | 0/100 [00:00<?, ?it/s]

Seg Tinku:   0%|          | 0/100 [00:00<?, ?it/s]

Seg Tunantada:   0%|          | 0/100 [00:00<?, ?it/s]

Seg Yaravi (Harawi):   0%|          | 0/100 [00:00<?, ?it/s]

Tracks procesados: 1000 | con segmentos: 1000
Segmentos 5s en: C:\Users\RICHI\Documents\Tesis\Data\Andina5seg\segmentacion5seg
